# Rennaigan backend on Colab

Runs the five detector services and the gateway on a Colab GPU and gives you a public
`https://….trycloudflare.com` address to paste into the UI (**Settings → ML Gateway → Gateway URL**).

Before running: **Runtime → Change runtime type → T4 GPU**. Then **Runtime → Run all**.
The first run takes about 15 minutes (model downloads). Keep this tab open while you use the UI.

If a cell fails, copy its full output; the self-test cell near the end reports each detector separately.

In [ ]:
#@title 1. Get the code
REPO = "https://github.com/Laxitha/Rennaigan.git"
BRANCH = "backend-gateway"   # change to "main" once the pull request is merged

import os, subprocess, torch
os.chdir("/content")
if not os.path.exists("/content/Rennaigan"):
    !git clone -q --depth 1 -b {BRANCH} {REPO} /content/Rennaigan
os.chdir("/content/Rennaigan")
!git pull -q
!git log --oneline -1
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to a T4 GPU")

In [ ]:
#@title 2. System tools and the model repositories
!apt-get -qq install -y libimage-exiftool-perl > /dev/null
!mkdir -p repos weights/sbi weights/univfd weights/trufor
repos = {
    "sbi": "https://github.com/mapooon/SelfBlendedImages.git",
    "univfd": "https://github.com/WisconsinAIVision/UniversalFakeDetect.git",
    "trufor": "https://github.com/grip-unina/TruFor.git",
    "lipforensics": "https://github.com/ahaliassos/LipForensics.git",
    "syncnet": "https://github.com/joonson/syncnet_python.git",
    "ssl_aasist": "https://github.com/TakHemlata/SSL_Anti-spoofing.git",
}
for name, url in repos.items():
    if not os.path.exists(f"repos/{name}"):
        !git clone -q --depth 1 {url} repos/{name}
!ls repos

In [ ]:
#@title 3. Python packages for the image, video, motion and metadata services
!pip install -q fastapi uvicorn python-multipart httpx pyyaml gdown yacs timm \
    efficientnet_pytorch open_clip_torch face_alignment scikit-image \
    python_speech_features "scenedetect[opencv]" onnxruntime-gpu
!pip install -q insightface
# Installed last and separately: these pin older numpy/protobuf and may print resolver warnings.
!pip install -q "mediapipe<0.11"
import importlib
for mod in ["efficientnet_pytorch", "open_clip", "insightface", "face_alignment", "mediapipe", "scenedetect", "yacs"]:
    try:
        importlib.import_module(mod); print("ok     ", mod)
    except Exception as e:
        print("MISSING", mod, "->", type(e).__name__, e)

In [ ]:
#@title 4. Separate Python 3.10 environment for the audio service (fairseq does not install on newer Python)
AUDIO_ENV = "/content/audio-env"
FAIRSEQ = "/content/Rennaigan/repos/ssl_aasist/fairseq-a54021305d6b3c4c5959ac9395135f63202db8f1"
if not os.path.exists(f"{AUDIO_ENV}/bin/python"):
    !pip install -q uv
    !uv venv -q {AUDIO_ENV} --python 3.10 --seed
    # pip 24.0: later versions reject the old omegaconf/hydra metadata that this fairseq commit needs
    !{AUDIO_ENV}/bin/python -m pip install -q "pip==24.0" "setuptools<70" wheel cython "numpy==1.23.5"
    !{AUDIO_ENV}/bin/pip install -q torch==2.1.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu121
    !cd {FAIRSEQ} && {AUDIO_ENV}/bin/pip install -q --editable ./
    !{AUDIO_ENV}/bin/pip install -q "numpy==1.23.5" soundfile librosa matplotlib fastapi uvicorn python-multipart pydantic pyyaml
!{AUDIO_ENV}/bin/python -c "import fairseq, torch, soundfile, librosa, fastapi; print('audio env ok: torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
#@title 5. Model weights
import glob, shutil

def have(path, min_mb=0.001):
    return os.path.exists(path) and os.path.getsize(path) > min_mb * 1e6

# UnivFD: the classifier weight ships inside its repository
if not have("weights/univfd/fc_weights.pth"):
    shutil.copy("repos/univfd/pretrained_weights/fc_weights.pth", "weights/univfd/fc_weights.pth")

# SBI (face swap), FF-c23 checkpoint
if not have("weights/sbi/FFc23.tar", 50):
    !gdown -q 1X0-NYT8KPursLZZdxduRQju6E52hauV0 -O weights/sbi/FFc23.tar

# TruFor (splicing)
if not have("weights/trufor/trufor.pth.tar", 50):
    !wget -q https://www.grip.unina.it/download/prog/TruFor/TruFor_weights.zip -O /content/trufor.zip
    !unzip -q -o /content/trufor.zip -d /content/trufor_unzipped
    found = glob.glob("/content/trufor_unzipped/**/trufor.pth.tar", recursive=True)
    if found: shutil.move(found[0], "weights/trufor/trufor.pth.tar")

# LipForensics
if not have("weights/lipforensics_ff.pth", 20):
    !gdown -q 1wfZnxZpyNd5ouJs0LjVls7zU0N_W73L7 -O weights/lipforensics_ff.pth

# SyncNet and its face detector
if not have("repos/syncnet/data/syncnet_v2.model", 10):
    !cd repos/syncnet && sh download_model.sh > /dev/null 2>&1

# XLS-R 300M backbone for the voice detector
if not have("weights/xlsr2_300m.pt", 1000):
    !wget -q https://dl.fbaipublicfiles.com/fairseq/wav2vec/xlsr2_300m.pt -O weights/xlsr2_300m.pt

# SSL-AASIST voice anti-spoofing checkpoint (shared Drive folder from the SSL_Anti-spoofing README)
if not have("weights/best_SSL_model_DF.pth", 100):
    !gdown -q --folder https://drive.google.com/drive/folders/1c4ywztEVlYVijfwbGLl9OEa1SNtFKppB -O /content/aasist_dl
    files = sorted(glob.glob("/content/aasist_dl/**/*.pth", recursive=True))
    print("AASIST files in the Drive folder:", [os.path.basename(f) for f in files])
    pick = next((f for f in files if "DF" in os.path.basename(f)), files[0] if files else None)
    if pick:
        shutil.copy(pick, "weights/best_SSL_model_DF.pth"); print("using", os.path.basename(pick))

print()
for label, path, mb in [
    ("SBI", "weights/sbi/FFc23.tar", 50), ("UnivFD", "weights/univfd/fc_weights.pth", 0.001),
    ("TruFor", "weights/trufor/trufor.pth.tar", 50), ("LipForensics", "weights/lipforensics_ff.pth", 20),
    ("SyncNet", "repos/syncnet/data/syncnet_v2.model", 10), ("SyncNet face detector", "repos/syncnet/detectors/s3fd/weights/sfd_face.pth", 10),
    ("XLS-R", "weights/xlsr2_300m.pt", 1000), ("SSL-AASIST", "weights/best_SSL_model_DF.pth", 100),
]:
    size = os.path.getsize(path) / 1e6 if os.path.exists(path) else 0
    print(f"{'ok     ' if size > mb else 'MISSING'} {label:<22} {path}  ({size:.0f} MB)")
print("\nIf a Google Drive file is MISSING (Drive rate-limits popular files), download it in your browser"
      " from the link in that model's README and upload it to the path shown, then re-run this cell.")

In [ ]:
#@title 6. Start the detector services and the gateway
import time, requests
os.chdir("/content/Rennaigan")
!pkill -f "uvicorn" ; pkill -f "start.py"
time.sleep(2)
env = dict(os.environ, RENNAIGAN_PYTHON_AUDIO="/content/audio-env/bin/python", PYTHONUNBUFFERED="1")
backend = subprocess.Popen(["python", "start.py"], env=env, stdout=open("/content/backend.log", "w"), stderr=subprocess.STDOUT)

health = None
for _ in range(120):
    time.sleep(3)
    try:
        health = requests.get("http://127.0.0.1:8010/health", timeout=3).json()
        if health["services_up"] == health["services_total"]: break
    except Exception:
        pass
!head -20 /content/backend.log
if health:
    print(f"\nservices up: {health['services_up']}/{health['services_total']}")
    for name, s in health["services"].items():
        print(f"  {'online ' if s['ok'] else 'OFFLINE'} {name}" + ("" if s["ok"] else f"   -> see: !tail -30 /content/Rennaigan/data/logs/{name}.log"))
    print("tools:", health["tools"])
else:
    print("Gateway did not start. Run:  !cat /content/backend.log")

In [ ]:
#@title 7. Self-test: every detector on a real talking-head clip
# The first call downloads CLIP and the face models, so this can take several minutes.
sample = "/content/Rennaigan/repos/syncnet/data/example.avi"
!ffmpeg -y -loglevel error -i {sample} -t 12 -c:v libx264 -pix_fmt yuv420p -c:a aac /content/test_clip.mp4
!ffmpeg -y -loglevel error -ss 2 -i {sample} -frames:v 1 -q:v 2 /content/test_frame.jpg
!ffmpeg -y -loglevel error -i {sample} -t 12 -vn -ar 16000 -ac 1 /content/test_voice.wav

def run(path):
    with open(path, "rb") as f:
        r = requests.post("http://127.0.0.1:8010/analyze", files={"file": (os.path.basename(path), f)}, timeout=1800)
    c = r.json()
    if "id" not in c:
        print(path, "->", r.status_code, c); return
    print(f"\n=== {os.path.basename(path)}  ->  {c['label']}, trust {c['trust_score']}, evidence {c['evidence_weight']:.0%}, {c['runtime_s']} s")
    print("   ", c["label_reason"])
    for name, m in c["modules"].items():
        print(f"  [{m['status']:<13}] {name:<9} {m.get('elapsed_s')} s  {(m['detail'] or '')[:300]}")
    models = {}
    for f in c["findings"]:
        models.setdefault((f["module"], f["model"], f["kind"]), []).append(f)
    for (module, model, kind), fs in models.items():
        top = max(fs, key=lambda f: f["score"])
        print(f"      {module}/{model:<14} {kind:<6} n={len(fs):<3} max={top['score']:.3f}  {top['note'][:110]}")
    print("   artifacts:", [a["name"] for a in c["artifacts"] if a["available"]])

for path in ["/content/test_frame.jpg", "/content/test_voice.wav", "/content/test_clip.mp4"]:
    run(path)
print("\nThis clip is a genuine recording: scores should be low. Lines marked unavailable/error need fixing; copy this output.")

In [ ]:
#@title 8. Public address for the UI
import re
if not os.path.exists("/content/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared
!pkill -f cloudflared
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8010", "--no-autoupdate"],
                          stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(40):
    time.sleep(1.5)
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if found:
        url = found.group(0); break
print("Gateway URL:", url or "not ready - run:  !cat /content/tunnel.log")
print("\nPaste it into the UI: Settings -> ML Gateway -> Gateway URL -> Save & Connect.")
print("Anyone with this address can upload to and delete from this gateway. It stops when this Colab session ends.")

In [ ]:
#@title Logs (run when something looks wrong)
!tail -15 /content/backend.log
for name in ["image", "video", "audio", "metadata", "motion"]:
    print(f"\n----- {name}")
    !tail -12 /content/Rennaigan/data/logs/{name}.log